# 한국형 Reasoning 모델 만들기 - Gemma 4 E4B + KOREAson YiSang

**환경**: Google Colab Pro (A100 또는 L4 GPU 권장)

**베이스 모델**: `unsloth/gemma-4-E4B-it` (8B params, 4B 활성)

**데이터셋**: `KOREAson/YiSang-HighQuality`

**방법**: QLoRA SFT (Unsloth)

---

## 사전 준비
1. HuggingFace **Write** 토큰 발급 (Settings → Access Tokens)
2. W&B API 키 복사 (wandb.ai → User Settings)
3. 런타임 → 런타임 유형 변경 → **A100** 또는 **L4 GPU** 선택

## STEP 1. 패키지 설치
설치 후 **런타임 → 세션 다시 시작** 한 번 눌러주세요.

In [ ]:
!pip install -q --upgrade unsloth
!pip install -q --upgrade transformers trl peft accelerate bitsandbytes datasets wandb

## STEP 2. HuggingFace & W&B 로그인

In [ ]:
from huggingface_hub import login
import wandb

HF_TOKEN = "hf_여기에_본인_토큰"
WANDB_KEY = "여기에_본인_WB_키"

login(HF_TOKEN)
wandb.login(key=WANDB_KEY)

## STEP 3. 모델 로드 (4bit 양자화)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

## STEP 4. LoRA 어댑터 부착

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## STEP 5. 데이터셋 로드 & 구조 확인

In [ ]:
from datasets import load_dataset

ds = load_dataset("KOREAson/YiSang-HighQuality", split="train")
print(ds)
print("---")
print(ds[0])

## STEP 6. 데이터 포맷팅

위 출력을 보고 컬럼명에 맞게 둘 중 하나를 사용하세요.

**A) `messages` 컬럼인 경우** (chat 형식)

In [ ]:
def formatting_func(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds = ds.map(formatting_func)
ds = ds.shuffle(seed=42).select(range(5000))  # 첫 시도는 5천개로
print(ds[0]["text"][:500])

**B) `instruction` / `output` 컬럼인 경우** (위 셀 대신 아래 사용)

In [ ]:
# def formatting_func(example):
#     msgs = [
#         {"role": "user", "content": example["instruction"]},
#         {"role": "assistant", "content": example["output"]},
#     ]
#     return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}
#
# ds = ds.map(formatting_func)
# ds = ds.shuffle(seed=42).select(range(5000))

## STEP 7. 학습 실행
5천 샘플 1 epoch: A100 약 30~60분 / L4 약 1.5~2시간 예상

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        output_dir = "outputs",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.03,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        bf16 = True,
        logging_steps = 10,
        save_steps = 200,
        save_total_limit = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 42,
        report_to = "wandb",
        run_name = "ko-reason-g4-e4b-test",
    ),
)

trainer.train()

## STEP 8. 모델 저장 & HF 업로드

In [ ]:
# LoRA 어댑터만 로컬 저장
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 베이스 모델과 머지해서 HF에 업로드 (16bit)
model.push_to_hub_merged(
    "본인HF아이디/ko-reason-g4-e4b-v1",
    tokenizer,
    save_method = "merged_16bit",
    token = HF_TOKEN,
)

## STEP 9. 추론 테스트

In [ ]:
FastLanguageModel.for_inference(model)

msgs = [{"role": "user", "content": "피보나치 수열의 10번째 항을 단계별로 구해줘."}]
inputs = tokenizer.apply_chat_template(
    msgs, return_tensors="pt", add_generation_prompt=True
).to("cuda")

out = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## STEP 10. 맥북 M5에서 로컬 추론 (옵션)

터미널에서 실행:

```bash
pip install mlx-lm
python -m mlx_lm.convert --hf-path 본인HF아이디/ko-reason-g4-e4b-v1 -q
python -m mlx_lm.generate --model mlx_model --prompt "안녕"
```

---

## 다음 단계
1. 5천 샘플로 파이프라인 검증 OK → 50k → 284k 전체로 확대
2. OOM 발생 시: `per_device_train_batch_size=1`, `max_seq_length=2048`로 낮추기
3. 풀 데이터셋(284k) 1 epoch는 A100에서 8~12시간 — 중간 체크포인트 push 필수